In [1]:
"""
Given strings s and t, return the minimum window 
substring of s that contains all characters of t.

Example:
  s = "ADOBECODEBANC", t = "ABC"
  Output: "BANC"

Pattern: sliding window with character frequency map
  Expand right until window is valid
  Contract left to minimise
"""

from collections import Counter

def minWindow(s: str, t: str) -> str:
    if not t or not s:
        return ""

    need    = Counter(t)     # chars needed
    have    = {}             # chars in window
    formed  = 0              # chars satisfying need
    required = len(need)     # distinct chars needed

    left    = 0
    best    = (float('inf'), 0, 0)  # (length, l, r)

    for right in range(len(s)):
        ch          = s[right]
        have[ch]    = have.get(ch, 0) + 1

        # Did this char satisfy its requirement?
        if ch in need and have[ch] == need[ch]:
            formed += 1

        # Contract window from left
        while formed == required:
            # Update best if smaller window
            if right - left + 1 < best[0]:
                best = (right - left + 1, left, right)

            # Remove leftmost char
            left_ch      = s[left]
            have[left_ch] -= 1
            if left_ch in need and \
               have[left_ch] < need[left_ch]:
                formed -= 1
            left += 1

    return "" if best[0] == float('inf') \
               else s[best[1]:best[2]+1]


# Test
cases = [
    ("ADOBECODEBANC", "ABC", "BANC"),
    ("a", "a", "a"),
    ("a", "aa", ""),
]
for s, t, expected in cases:
    result = minWindow(s, t)
    status = "✅" if result == expected else "❌"
    print(f"{status} minWindow({s!r}, {t!r})")
    print(f"   Expected: {expected!r}")
    print(f"   Got:      {result!r}\n")

print("Time:  O(|s| + |t|)")
print("Space: O(|t|)")
print()
print("Pattern learned:")
print("  Two pointers (left, right)")
print("  Expand right → contract left")
print("  Track 'formed' count vs 'required'")

✅ minWindow('ADOBECODEBANC', 'ABC')
   Expected: 'BANC'
   Got:      'BANC'

✅ minWindow('a', 'a')
   Expected: 'a'
   Got:      'a'

✅ minWindow('a', 'aa')
   Expected: ''
   Got:      ''

Time:  O(|s| + |t|)
Space: O(|t|)

Pattern learned:
  Two pointers (left, right)
  Expand right → contract left
  Track 'formed' count vs 'required'


In [2]:
"""
Merge k sorted linked lists into one sorted list.

Pattern: min-heap
  Push first node of each list into heap
  Pop minimum, push its next node
  
Why heap: extracting minimum from k lists
  Naive: O(nk) — scan all list heads each time
  Heap:  O(n log k) — heap maintains order
"""

import heapq
from typing import Optional, List

class ListNode:
    def __init__(self, val=0, next=None):
        self.val  = val
        self.next = next

    def __lt__(self, other):
        # Needed for heap comparison
        return self.val < other.val


def mergeKLists(
        lists: List[Optional[ListNode]]
    ) -> Optional[ListNode]:

    heap   = []
    dummy  = ListNode(0)
    curr   = dummy

    # Push head of each non-empty list
    for node in lists:
        if node:
            heapq.heappush(heap, node)

    while heap:
        node  = heapq.heappop(heap)
        curr.next = node
        curr      = curr.next
        if node.next:
            heapq.heappush(heap, node.next)

    return dummy.next


# Build test lists
def make_list(vals):
    dummy = ListNode(0)
    curr  = dummy
    for v in vals:
        curr.next = ListNode(v)
        curr      = curr.next
    return dummy.next

def list_to_arr(node):
    result = []
    while node:
        result.append(node.val)
        node = node.next
    return result


lists1   = [make_list([1,4,5]),
             make_list([1,3,4]),
             make_list([2,6])]
result1  = list_to_arr(mergeKLists(lists1))
expected = [1,1,2,3,4,4,5,6]
print(f"{'✅' if result1 == expected else '❌'} "
      f"mergeKLists")
print(f"   Expected: {expected}")
print(f"   Got:      {result1}\n")

print("Time:  O(n log k)  — n total nodes, k lists")
print("Space: O(k)        — heap size")
print()
print("The heap template:")
print("  heapq.heappush(heap, node)")
print("  node = heapq.heappop(heap)")
print("  Heap maintains smallest-first order")

✅ mergeKLists
   Expected: [1, 1, 2, 3, 4, 4, 5, 6]
   Got:      [1, 1, 2, 3, 4, 4, 5, 6]

Time:  O(n log k)  — n total nodes, k lists
Space: O(k)        — heap size

The heap template:
  heapq.heappush(heap, node)
  node = heapq.heappop(heap)
  Heap maintains smallest-first order


In [3]:
"""
Design a data structure that supports:
  addNum(num): add integer to structure
  findMedian():  return median of all elements

Pattern: two heaps
  max-heap: lower half of numbers
  min-heap: upper half of numbers

  Invariant: len(max_heap) == len(min_heap)
             OR len(max_heap) == len(min_heap) + 1

  Median: max_heap[0]              if odd total
          (max_heap[0]+min_heap[0])/2  if even total
"""

import heapq

class MedianFinder:
    def __init__(self):
        # Max heap for lower half
        # Python has min-heap only → negate values
        self.lo = []   # max-heap (negated)
        # Min heap for upper half
        self.hi = []   # min-heap

    def addNum(self, num: int) -> None:
        # Always push to max-heap first
        heapq.heappush(self.lo, -num)

        # Balance: lo's max ≤ hi's min
        if self.hi and (-self.lo[0] > self.hi[0]):
            heapq.heappush(
                self.hi,
                -heapq.heappop(self.lo)
            )

        # Size balance: lo has at most 1 more
        if len(self.lo) > len(self.hi) + 1:
            heapq.heappush(
                self.hi,
                -heapq.heappop(self.lo)
            )
        elif len(self.hi) > len(self.lo):
            heapq.heappush(
                self.lo,
                -heapq.heappop(self.hi)
            )

    def findMedian(self) -> float:
        if len(self.lo) > len(self.hi):
            return float(-self.lo[0])
        return (-self.lo[0] + self.hi[0]) / 2.0


# Test
mf = MedianFinder()
stream = [1, 2, 3, 4, 5]
for num in stream:
    mf.addNum(num)
    print(f"  Added {num}: median = {mf.findMedian()}")

print()
print("Key insight: two heaps maintain sorted halves")
print("  lo (max-heap): [1, 2, 3] → top=3")
print("  hi (min-heap): [4, 5]    → top=4")
print("  median = 3 (odd) or (3+4)/2=3.5 (even)")

  Added 1: median = 1.0
  Added 2: median = 1.5
  Added 3: median = 2.0
  Added 4: median = 2.5
  Added 5: median = 3.0

Key insight: two heaps maintain sorted halves
  lo (max-heap): [1, 2, 3] → top=3
  hi (min-heap): [4, 5]    → top=4
  median = 3 (odd) or (3+4)/2=3.5 (even)


In [4]:
"""
Given beginWord, endWord, and wordList,
return the number of words in the shortest
transformation sequence from beginWord to endWord.

Each step: change exactly one letter.
Each intermediate word must be in wordList.

Pattern: BFS on implicit graph
  Each word = node
  Edge exists if words differ by one letter
  BFS finds shortest path
"""

from collections import deque, defaultdict

def ladderLength(beginWord: str,
                  endWord: str,
                  wordList: list) -> int:

    word_set = set(wordList)
    if endWord not in word_set:
        return 0

    # BFS
    queue = deque([(beginWord, 1)])  # (word, steps)
    visited = {beginWord}

    while queue:
        word, steps = queue.popleft()

        # Try changing each character
        for i in range(len(word)):
            for c in 'abcdefghijklmnopqrstuvwxyz':
                new_word = word[:i] + c + word[i+1:]

                if new_word == endWord:
                    return steps + 1

                if (new_word in word_set and
                        new_word not in visited):
                    visited.add(new_word)
                    queue.append((new_word, steps+1))

    return 0   # No path found


# Test
print(ladderLength(
    "hit", "cog",
    ["hot","dot","dog","lot","log","cog"]
))  # Expected: 5
# hit → hot → dot → dog → cog

print(ladderLength(
    "hit", "cog",
    ["hot","dot","dog","lot","log"]
))  # Expected: 0 (cog not in list)

print("\nTime:  O(M² × N)  M=word length, N=wordList size")
print("Space: O(M² × N)")
print()
print("Pattern: BFS on IMPLICIT graph")
print("  Don't build graph explicitly")
print("  Generate neighbours on the fly")
print("  BFS guarantees shortest path")

5
0

Time:  O(M² × N)  M=word length, N=wordList size
Space: O(M² × N)

Pattern: BFS on IMPLICIT graph
  Don't build graph explicitly
  Generate neighbours on the fly
  BFS guarantees shortest path


In [5]:
patterns_complete = """
=== 7 DSA Patterns — Complete ===

1. Arrays & Hashmaps    ✅ 12 problems
   → Two Sum, Group Anagrams, Top K Frequent

2. Sliding Window        ✅ 5 problems  ← completed today
   → Min Window Substring, Longest Substring

3. Stack & Queue         ✅ 5 problems
   → Valid Parentheses, Min Stack

4. Trees                 ✅ 6 problems
   → Level Order BFS, Max Depth, LCA

5. Linked Lists          ✅ 5 problems
   → Merge K Sorted ← completed today
   → Reverse, Detect Cycle

6. Dynamic Programming   ✅ 6 problems
   → Climb Stairs, Coin Change, LCS

7. Graphs                ✅ 8 problems  ← completed today
   → Word Ladder, Dijkstra, Topo Sort
   → BFS, DFS, Union-Find

Heap problems (bonus):  ✅ 2 problems
   → Find Median ← completed today
   → Merge K Lists ← completed today

Total: 52 problems across all patterns.

What 52 problems unlocks:
  → Can pass 80% of Indian company OA rounds
  → Sufficient for Tier-1 company first rounds
    (Flipkart, Atlassian, Adobe, etc.)
  → Sufficient for most ML engineer internships
  
Next DSA phase (after exams):
  → Company-specific grinding (50+ more problems)
  → System design introduction
  → Mock interviews
"""
print(patterns_complete)


=== 7 DSA Patterns — Complete ===

1. Arrays & Hashmaps    ✅ 12 problems
   → Two Sum, Group Anagrams, Top K Frequent

2. Sliding Window        ✅ 5 problems  ← completed today
   → Min Window Substring, Longest Substring

3. Stack & Queue         ✅ 5 problems
   → Valid Parentheses, Min Stack

4. Trees                 ✅ 6 problems
   → Level Order BFS, Max Depth, LCA

5. Linked Lists          ✅ 5 problems
   → Merge K Sorted ← completed today
   → Reverse, Detect Cycle

6. Dynamic Programming   ✅ 6 problems
   → Climb Stairs, Coin Change, LCS

7. Graphs                ✅ 8 problems  ← completed today
   → Word Ladder, Dijkstra, Topo Sort
   → BFS, DFS, Union-Find

Heap problems (bonus):  ✅ 2 problems
   → Find Median ← completed today
   → Merge K Lists ← completed today

Total: 52 problems across all patterns.

What 52 problems unlocks:
  → Can pass 80% of Indian company OA rounds
  → Sufficient for Tier-1 company first rounds
    (Flipkart, Atlassian, Adobe, etc.)
  → Sufficient for 